# ◈ STARK INDUSTRIES · NEURAL TRANSLATION ARRAY
### `v9.2` · Gemma3:27b · Story State Engine v4.0 · 2-Pass · ToneGuard · DeadBoringBase

```
  ╔══════════════════════════════════════════════════════════════════╗
  ║  J.A.R.V.I.S  —  Just A Rather Very Intelligent System          ║
  ║  Hinglish Literary Translation Engine v9.0                       ║
  ║  Model: gemma3:27b  ·  2-Pass Architecture  ·  T4 (15GB)        ║
  ╚══════════════════════════════════════════════════════════════════╝
```

| Step | Cell | Mission Directive |
|------|------|-------------------|
| **①** | 2 | Install dependencies |
| **②** | 4–5 | Boot Ollama · Pull gemma3:27b (~17 GB, Q4_K_M) |
| **③** | 7 | Upload source `.txt` file |
| **④** | 9–10 | Configure parameters |
| **⑤** | 12 | Story State Engine v4.0 |
| **⑥** | 14 | Load Translation Engine v9.1 |
| **⑦** | 16 | **⚡ Execute translation** |
| **⑧** | 18 | Download output + `state.json` |

> **v9.0** — 2-Pass Architecture: Step1=Semantic Base (faithful) → Step2=Hinglish Style Filter (safe)  
> Story State Engine v4.0: `character_speech_styles` replaces heavy summaries · Lean context block · Strict Fidelity Rule

> **Why 2-Pass?** Single-pass forces the model to optimize accuracy + style + tone + continuity simultaneously → random tradeoffs. Two-pass separates concerns: Step1 builds a stable semantic base, Step2 styles it safely.


## ⚡ Step 1 — Install Dependencies
Run once per Colab session.

In [1]:
!pip install -q torch transformers accelerate sentencepiece ollama ipywidgets
print('✅ All dependencies installed!')
from IPython.display import display, HTML
display(HTML('''
<div style="background:#070710;border:2px solid #c0392b;border-radius:8px;
            padding:12px 18px;font-family:'Courier New',monospace;margin-top:10px;">
  <div style="color:#c0392b;font-size:1.1em;font-weight:bold;letter-spacing:2px;">[ DEPENDENCIES INSTALLED ]</div>
  <div style="color:#4CAF50;font-size:0.85em;margin-top:4px;">torch · transformers · accelerate · sentencepiece · ollama · ipywidgets</div>
</div>
'''))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.8 MB/s eta 0:00:00


KeyboardInterrupt: 

## 🦙 Step 2a — Boot Ollama Server
Run Cell 4, then Cell 5 to pull **gemma3:27b** (~17 GB, Q4_K_M).

> ⚠️ T4 has 15 GB VRAM and ~12 GB RAM. gemma3:27b Q4_K_M needs ~17 GB — **Ollama uses CPU offloading for layers that don't fit, which is slower but works.** Expect ~45-90 seconds per chunk vs ~20-30s for 9B models.

In [ ]:
import subprocess, time, os
from IPython.display import display, HTML

display(HTML('<div style="background:linear-gradient(135deg,#0a0a0f,#1a0505);border:2px solid #c0392b;'
            'border-radius:8px;padding:14px 18px;font-family:Courier New,monospace;">'
            '<div style="color:#c0392b;font-size:1.2em;font-weight:bold;letter-spacing:3px;">◈ OLLAMA BOOT SEQUENCE</div>'
            '<div style="color:#FFD700;font-size:0.82em;margin-top:4px;">Installing Ollama runtime — gemma3:27b edition</div></div>'))

!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

print('\n🚀 Starting Ollama server...')
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
# Allow more VRAM for gemma3:27b layers
os.environ['OLLAMA_GPU_OVERHEAD'] = '512000000'  # 512 MB overhead
subprocess.Popen(['/usr/local/bin/ollama', 'serve'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)

try:
    import ollama; ollama.list()
    display(HTML('<div style="background:#0a0f0a;border:2px solid #4CAF50;border-radius:6px;'
                'padding:10px 18px;font-family:Courier New,monospace;margin-top:8px;">'
                '<span style="color:#4CAF50;font-weight:bold;">[OK] ARC REACTOR STABLE — Ollama server operational.</span><br>'
                '<span style="color:#888;font-size:0.82em;">Run Cell 5 to pull gemma3:27b (~17 GB — takes 10-25 min).</span></div>'))
except Exception as e:
    print(f'⚠️ Server may still be starting: {e} — wait 5s and re-run.')

In [ ]:
import ollama

MODEL_NAME = 'gemma3:27b'
print(f'📥 Pulling {MODEL_NAME} (Q4_K_M ~17 GB) — this takes 10-25 minutes...')
print('   Go make chai ☕')

try:
    current_digest = ''
    for progress in ollama.pull(MODEL_NAME, stream=True):
        digest = progress.get('digest', '')
        if digest != current_digest and current_digest: print()
        current_digest = digest
        status = progress.get('status', '')
        if 'completed' in progress and 'total' in progress:
            pct = (progress['completed'] / progress['total'] * 100) if progress['total'] else 0
            bar_len = int(pct / 2)
            bar = '█' * bar_len + '░' * (50 - bar_len)
            print(f'\r   [{bar}] {pct:.1f}%', end='', flush=True)
        else:
            print(f'\r   {status}', end='', flush=True)
    print(f'\n\n✅ {MODEL_NAME} ready!')
    for m in ollama.list().get('models', []):
        print(f"   • {m.get('name')} ({m.get('size',0)/(1024**3):.2f} GB)")
except Exception as e:
    print(f'\n❌ Pull failed: {e}')
    print('   Make sure Ollama server is running (Cell 4).')

## 📤 Step 3 — Upload Source File
Upload the `.txt` book file.

In [ ]:
from IPython.display import display, HTML
from google.colab import files
import re

display(HTML('<div style="background:linear-gradient(135deg,#0a0a0f,#1a0505);border:2px solid #c0392b;'
            'border-radius:8px;padding:14px 18px;font-family:Courier New,monospace;margin-bottom:8px;">'
            '<div style="color:#c0392b;font-size:1.15em;font-weight:bold;letter-spacing:2px;">◈ FILE UPLINK</div>'
            '<div style="color:#FFD700;font-size:0.85em;margin-top:4px;">Select your .txt file (any chapter, any book).</div></div>'))

uploaded = files.upload()
UPLOADED_FILE = list(uploaded.keys())[0]

with open(UPLOADED_FILE, 'r', encoding='utf-8') as f:
    _raw = f.read()

# Auto-strip TranslateGemma / JARVIS pipeline headers (ch2+ files have these)
_cleaned = re.sub(
    r'^TRANSLATED TO ENGLISH\s*[=\-]{10,}.*?[=\-]{10,}+','', _raw, flags=re.DOTALL
).strip()
if _cleaned != _raw:
    with open(UPLOADED_FILE, 'w', encoding='utf-8') as f:
        f.write(_cleaned)
    print(f'[OK] Pipeline header stripped automatically.')
    print(f'[OK] English source pre-processing will run before translation (German residuals, etc.)')

_words = len(_cleaned.split())
print(f'\n✅ File ready: {UPLOADED_FILE}')
print(f'   {_words:,} words | {len(_cleaned):,} chars')
print(f'   Preview: {_cleaned[:180].strip()!r}...')


## ⚙️ Step 4 — Configure Parameters

| Parameter | Default | Notes |
|-----------|---------|-------|
| Chunk size | **450 words** | Gemma3:27b handles larger chunks well |
| Overlap | **80 words** | More overlap = better flow |
| num_ctx | **8192** | Gemma3 handles long context cleanly |

> Gemma3:27b runs slower than 9B models (~60-90s/chunk on T4). For a 50,000 word book (~111 chunks at 450w), expect **2-3 hours**.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

TIER_OPTIONS = {
    'BASIC  — Fast, clean Hindi': 'BASIC',
    'INTERMEDIATE  — Balanced': 'INTERMEDIATE',
    'ADVANCED  — Full Millennial Hinglish (recommended)': 'ADVANCED',
}
tier_dropdown = widgets.Dropdown(
    options=list(TIER_OPTIONS.keys()),
    value='ADVANCED  — Full Millennial Hinglish (recommended)',
    description='Quality Tier:', style={'description_width':'initial'},
    layout=widgets.Layout(width='500px')
)
chunk_slider = widgets.IntSlider(
    value=350, min=150, max=600, step=25,  # FIX 6: 350 default for dense literary prose
    description='Chunk size (words):', style={'description_width':'initial'},
    layout=widgets.Layout(width='500px')
)
overlap_slider = widgets.IntSlider(
    value=80, min=0, max=150, step=10,
    description='Overlap (words):', style={'description_width':'initial'},
    layout=widgets.Layout(width='500px')
)
num_ctx_slider = widgets.IntSlider(
    value=8192, min=4096, max=12288, step=1024,
    description='num_ctx (tokens):', style={'description_width':'initial'},
    layout=widgets.Layout(width='500px')
)
LANGUAGE_OPTIONS = {
    'Hinglish (Roman script Hindi — DEFAULT)': 'hinglish',
    'Hindi (Devanagari)': 'hin_Deva',
    'Bengali': 'ben_Beng',
    'Tamil': 'tam_Taml',
    'Telugu': 'tel_Telu',
    'Marathi': 'mar_Deva',
    'Gujarati': 'guj_Gujr',
}
lang_dropdown = widgets.Dropdown(
    options=list(LANGUAGE_OPTIONS.keys()),
    value='Hinglish (Roman script Hindi — DEFAULT)',
    description='Target Language:', style={'description_width':'initial'},
    layout=widgets.Layout(width='500px')
)
display(HTML('<div style="background:#0a0a0f;border:2px solid #c0392b;border-radius:8px;'
            'padding:12px 18px;font-family:Courier New,monospace;margin-bottom:10px;">'
            '<div style="color:#c0392b;font-weight:bold;letter-spacing:2px;">◈ MISSION PARAMETERS — gemma3:27b v8.0</div>'
            '<div style="color:#888;font-size:0.78em;margin-top:4px;">Story State Engine: updates BEFORE every chunk | temp=0.7 top_k=40 top_p=0.9</div>'
            '</div>'))
display(tier_dropdown, chunk_slider, overlap_slider, num_ctx_slider, lang_dropdown)
print('\n💡 Adjust then run Cell 10 to lock config.')

In [ ]:
import os
MODEL         = 'gemma3:27b'
CHUNK_SIZE    = chunk_slider.value
OVERLAP_WORDS = overlap_slider.value
TRANSLATION_TIER = TIER_OPTIONS[tier_dropdown.value]
NUM_CTX       = num_ctx_slider.value
TARGET_LANG   = LANGUAGE_OPTIONS[lang_dropdown.value]
OUTPUT_DIR    = './translation_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Configuration locked:')
print(f'   🤖 Model          : {MODEL}')
print(f'   🎯 Quality Tier   : {TRANSLATION_TIER}')
print(f'   🌐 Target Language: {TARGET_LANG}')
print(f'   📦 Chunk Size     : {CHUNK_SIZE} words')
print(f'   🔀 Overlap        : {OVERLAP_WORDS} words')
print(f'   🧠 num_ctx        : {NUM_CTX} tokens')
print()
print('ℹ️  Gemma3:27b Parameters (fixed):')
print('   temperature=0.70  top_k=40  top_p=0.90  repeat_penalty=1.1')
print('   num_predict=2048  (generous for 27B quality)')
print('   Story State: updates BEFORE each chunk')

## 🧠 Step 5 — Story State Engine v4.0

**What changed from v3.0:**
- ❌ Removed heavy `story_so_far` summaries (model was ignoring them under prompt load)
- ✅ Added `character_speech_styles` — compact per-character tone descriptors
- ✅ `build_context_prompt()` now outputs only **last 1 summary** + **char speech styles**
- ✅ Context block is lean: fits in ~300 chars vs ~2000 chars before

**Manual setup (before running):**
```python
add_vocab_entry('the game is afoot', 'khel shuru ho gaya')
update_character('Holmes', role='eccentric detective', address='aap',
                 speech_style='calm, precise, analytical — never informal')
```
**Resume a book:** `load_state('path/to/state.json')`


In [ ]:
import re, json, os
from collections import Counter
from pathlib import Path

# ════════════════════════════════════════════════════════════════════════
# STORY STATE ENGINE v4.0  — JARVIS v9.0
# Changes from v3.0:
#   - Added character_speech_styles dict (replaces heavy summary injection)
#   - story_so_far capped at 1 entry (only last chunk) — was 6
#   - build_context_prompt() slimmed: char tone + last summary only
#   - Removed long summaries from context block (model was ignoring them)
# ════════════════════════════════════════════════════════════════════════

STORY_STATE = {
    'book_title':            '',
    'genre':                 '',
    'characters':            {},
    'character_speech_styles': {},  # NEW v4.0: per-char compact tone descriptor
    'current_setting':       '',
    'dominant_tone':         'DAILY_LIFE',
    'established_vocab':     {},
    'story_so_far':          [],   # NOW capped at 1 entry (only last chunk)
    'chunk_count':           0,
    'total_chunks':          0,
}

# ── Genre vocabulary defaults ─────────────────────────────────────────
GENRE_VOCAB = {
    'mystery/detective': {
        'deduction': 'deduction', 'the evidence': 'saboot', 'clue': 'suraag',
        'inspector': 'inspector sahab', 'constable': 'constable',
        'the case': 'case', 'motive': 'wajah', 'alibi': 'alibi',
        'the suspect': 'shak wala insaan', 'investigation': 'tapaas',
        'the mystery': 'raaz', 'solution': 'hal',
    },
    'gothic/horror': {
        'the creature': 'woh ajeeb makhluq', 'the monster': 'woh rakshas',
        'the castle': 'woh purana qila', 'the darkness': 'gehri raat ka andhera',
        'the grave': 'qabr', 'supernatural': 'anokha', 'terror': 'khouf',
        'the curse': 'baddua', 'the shadow': 'parchhaayi', 'the ghost': 'bhoot',
    },
    'romance/social': {
        'the marriage': 'shaadi', 'the proposal': 'shaadi ka proposal',
        'society': 'samaj ke log', 'fortune': 'daulat',
        'an eligible man': 'ek achha rishta', 'reputation': 'izzat',
        'the ball': 'dance party', 'the estate': 'haveli aur zameen',
        'governess': 'ghar ki teacher',
    },
    'adventure': {
        'the quest': 'woh mission', 'the battle': 'jung', 'the sword': 'talwar',
        'the enemy': 'dushman', 'the captain': 'captain sahab',
        'the ship': 'jahaaz', 'treasure': 'khazana', 'the expedition': 'safar',
    },
    'folk tale / fairy tale': {
        'the king': 'raja sahab', 'the queen': 'rani sahiba',
        'the prince': 'rajkumar', 'the princess': 'rajkumari',
        'the witch': 'daayan', 'the forest': 'jaangal', 'the village': 'gaon',
        'the spell': 'jadoo', 'happily ever after': 'phir woh khushi-khushi rehne lage',
    },
    'literary fiction': {},
    'general fiction':  {},
}

_STOP = {
    'The','This','That','These','Those','There','Here','When','Where','What',
    'Which','Who','How','Why','And','But','Or','For','As','At','By','In','Of',
    'On','To','It','He','She','We','They','My','Your','Mr','Mrs','Miss','Dr',
    'Sir','Then','Now','Just','Well','Very','Good','Little','Old','New','First',
    'Last','Next','Same','Even','Still','Again','Only','Always','Never','Every',
    'After','Before','With','About','Over','Into','From','Back','Down','Up','Out',
}


def _detect_genre(text):
    t = text.lower()
    if any(w in t for w in ['detective','clue','mystery','suspect','murder','crime','inspector']):
        return 'mystery/detective'
    if any(w in t for w in ['monster','vampire','ghost','horror','terror','creature','curse']):
        return 'gothic/horror'
    if any(w in t for w in ['love','proposal','marriage','society','fortune','darling','romance']):
        return 'romance/social'
    if any(w in t for w in ['king','queen','prince','princess','witch','spell','once upon']):
        return 'folk tale / fairy tale'
    if any(w in t for w in ['sword','battle','quest','army','expedition','treasure']):
        return 'adventure'
    return 'literary fiction'


def _extract_chars(text):
    titled = re.findall(
        r'(?:Mr\.?|Mrs\.?|Miss|Dr\.?|Sir|Lord|Lady|Captain|Inspector|'
        r'Professor|Colonel|Major|General|Count|Countess|Father|Mother)\s+'
        r'[A-Z][a-z]{2,}(?:\s+[A-Z][a-z]{2,})?', text)
    singles = re.findall(r'\b[A-Z][a-z]{2,}(?:\s+[A-Z][a-z]{2,})?\b', text)
    counts  = Counter(singles + titled)
    return [n for n,c in counts.most_common(20)
            if c >= 2 and n not in _STOP and len(n.split()[-1]) > 2][:8]


def _extract_setting(text):
    m = re.search(
        r'(?:at|in|inside|within|outside|near|entered|through|arrived at|left)\s+'
        r'(?:the\s+)?([A-Z][A-Za-z\s]{2,30}'
        r'(?:Street|Road|House|Hall|Room|Study|Garden|Park|Inn|Hotel|'
        r'Station|Office|Club|Square|Lane|Bridge|Castle|Manor|Lodge|'
        r'Village|Court|Tower|Library|Cottage|Drawing.?Room|Sitting.?Room))',
        text)
    return m.group(1).strip()[:60] if m else ''


def _english_fallback(english_chunk):
    """Best-effort 1-sentence English summary from source text."""
    sents = re.split(r'(?<=[.!?])\s+', english_chunk.strip())
    for s in sents:
        if len(s.split()) > 6:
            return s.strip()[:120]
    return ' '.join(english_chunk.split()[:25])


# ── NEW v4.0: speech style inference ─────────────────────────────────
_SPEECH_PATTERNS = {
    'formal':     ['I beg your pardon','I must inform','I am afraid','I assure you',
                   'It is my duty','permit me','I venture to','I should be'],
    'analytical': ['the evidence','it is clear','I observe','I deduce','therefore',
                   'logically','it follows','the facts show'],
    'anxious':    ['I fear','what if','surely not','I cannot bear','what shall',
                   'I dread','what will become'],
    'commanding': ['you will','do it at once','I demand','see to it','at once',
                   'immediately','I order'],
    'gentle':     ['my dear','pray','I hope','would you','I beg','if you please',
                   'perhaps you might'],
    'sarcastic':  ['how charming','how delightful','I am sure','how very',
                   'how convenient','how extraordinary'],
}

def _infer_speech_style(char_name, text):
    """Scan dialogue attributed to char_name and infer compact style descriptor."""
    # Find lines spoken by this character
    pattern = re.compile(
        r'["\u201c\u201d][^"\u201c\u201d]{5,200}["\u201c\u201d]'
        r'.*?' + re.escape(char_name.split()[-1]),
        re.IGNORECASE
    )
    dialogue_text = ' '.join(pattern.findall(text)).lower()
    if not dialogue_text:
        return ''
    scores = {style: sum(1 for kw in kws if kw in dialogue_text)
              for style, kws in _SPEECH_PATTERNS.items()}
    detected = [s for s, sc in scores.items() if sc > 0]
    return ', '.join(detected[:3]) if detected else 'natural'


def update_story_state(english_chunk, hinglish_chunk, chunk_idx,
                       scene_type='DAILY_LIFE', plot_note_en=''):
    global STORY_STATE
    if chunk_idx == 1 and not STORY_STATE['genre']:
        set_genre(_detect_genre(english_chunk))

    # Auto-discover characters
    for name in _extract_chars(english_chunk):
        if name not in STORY_STATE['characters']:
            STORY_STATE['characters'][name] = {'role':'character','address':'aap','notes':''}

    # NEW v4.0: Update speech styles from each chunk's dialogue
    for name in STORY_STATE['characters']:
        inferred = _infer_speech_style(name, english_chunk)
        if inferred and inferred != 'natural':
            existing = STORY_STATE['character_speech_styles'].get(name, '')
            if not existing:
                STORY_STATE['character_speech_styles'][name] = inferred

    s = _extract_setting(english_chunk)
    if s:
        STORY_STATE['current_setting'] = s
    STORY_STATE['dominant_tone'] = scene_type

    # v4.0: Keep ONLY the last 1 summary entry (was 6) — lighter context
    summary = (plot_note_en.strip() if plot_note_en and len(plot_note_en.split()) > 4
               else _english_fallback(english_chunk))
    STORY_STATE['story_so_far'] = [{'chunk': chunk_idx, 'summary': summary}]
    STORY_STATE['chunk_count'] = chunk_idx


def build_context_prompt():
    """
    v4.0 — LEAN context block injected before each chunk.
    Removed: heavy story summaries, long context blocks.
    Kept:    character speech styles, last 1 summary, vocab table.
    Rationale: model was ignoring long summaries under prompt load.
               Compact char-tone is far more useful than narrative recap.
    """
    s = STORY_STATE
    if s['chunk_count'] == 0 and not s['characters'] and not s['established_vocab']:
        return ''

    parts = ['=== STORY CONTEXT (sirf model ke liye — output mein mat daalo) ===']

    if s.get('book_title') or s.get('genre'):
        info = []
        if s.get('book_title'): info.append(f"Book: {s['book_title']}")
        if s.get('genre'):      info.append(f"Genre: {s['genre']}")
        parts.append(' | '.join(info))

    # v4.0: CHARACTER SPEECH STYLES — compact, actionable, replaces long summaries
    if s['characters']:
        parts.append('\nCHARACTERS — address form aur speech style consistent rakho:')
        for name, info in list(s['characters'].items())[:10]:
            style = s['character_speech_styles'].get(name, '')
            line = f"  • {name}"
            if info['role'] != 'character': line += f" ({info['role']})"
            line += f" | address: {info['address']}"
            if style:         line += f" | speech: {style}"
            if info['notes']: line += f" | {info['notes']}"
            parts.append(line)

    if s.get('current_setting'):
        parts.append(f"\nSETTING: {s['current_setting']}")

    if s.get('established_vocab'):
        parts.append('\nVOCAB TABLE — inhi words use karo:')
        for eng, hin in list(s['established_vocab'].items())[:15]:
            parts.append(f'  • "{eng}" → "{hin}"')

    # v4.0: Only last 1 summary (was up to 6) — just enough for continuity
    if s['story_so_far']:
        last = s['story_so_far'][-1]
        summary_txt = last['summary'] if isinstance(last, dict) else str(last)
        parts.append(f"\nLAST EVENT: {summary_txt}")

    parts.append('\n=== CONTEXT KHATAM ===')
    return '\n'.join(parts)


def save_state(filepath):
    try:
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(STORY_STATE, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print(f'[!] State save: {e}')


def load_state(filepath):
    global STORY_STATE
    try:
        if os.path.exists(filepath):
            with open(filepath, 'r', encoding='utf-8') as f:
                loaded = json.load(f)
            # v4.0 migration: add new fields if loading old state
            if 'character_speech_styles' not in loaded:
                loaded['character_speech_styles'] = {}
            STORY_STATE = loaded
            print(f'[OK] State loaded — chunk {STORY_STATE["chunk_count"]}, '
                  f'{len(STORY_STATE["characters"])} chars, '
                  f'{len(STORY_STATE["character_speech_styles"])} speech styles')
            return True
    except Exception as e:
        print(f'[!] load_state: {e}')
    return False


def _load_latest_state(output_dir='./translation_output'):
    """Load the most recent state file in output_dir. Used by resume_from='auto'."""
    import glob
    files = sorted(glob.glob(f'{output_dir}/state_*.json'))
    if files:
        return load_state(files[-1])
    return False


def reset_for_new_book(title='', genre=''):
    global STORY_STATE
    STORY_STATE = {
        'book_title': title, 'genre': '', 'characters': {},
        'character_speech_styles': {},
        'current_setting': '', 'dominant_tone': 'DAILY_LIFE',
        'established_vocab': {}, 'story_so_far': [],
        'chunk_count': 0, 'total_chunks': 0,
    }
    if genre:
        set_genre(genre)
    msg = f'[OK] Reset → "{title}"' if title else '[OK] State reset.'
    print(msg + (f' | Genre: {genre}' if genre else ''))


def set_genre(genre):
    STORY_STATE['genre'] = genre
    vocab = GENRE_VOCAB.get(genre, {})
    for eng, hin in vocab.items():
        if eng not in STORY_STATE['established_vocab']:
            STORY_STATE['established_vocab'][eng] = hin
    if vocab:
        print(f'[OK] Genre: {genre} | {len(vocab)} vocab defaults loaded')
    else:
        print(f'[OK] Genre: {genre}')


def set_book_title(title):
    STORY_STATE['book_title'] = title
    print(f'[OK] Book: "{title}"')


def add_vocab_entry(english, hinglish):
    if english.strip().lower() == hinglish.strip().lower():
        print(f'[SKIP] Same-value vocab ignored: "{english}" (no translation needed)')
        return
    STORY_STATE['established_vocab'][english] = hinglish
    print(f'[OK] Vocab: "{english}" → "{hinglish}"')


def update_character(name, role=None, address=None, notes=None, speech_style=None):
    """v4.0: Added speech_style parameter."""
    if name not in STORY_STATE['characters']:
        STORY_STATE['characters'][name] = {'role':'character','address':'aap','notes':''}
    e = STORY_STATE['characters'][name]
    if role:         e['role']    = role
    if address:      e['address'] = address
    if notes:        e['notes']   = notes
    if speech_style:
        STORY_STATE['character_speech_styles'][name] = speech_style
    print(f'[OK] Char: {name} → {e}')
    if speech_style:
        print(f'       speech_style: {speech_style}')


def print_state_summary():
    s = STORY_STATE
    print('\n=== STORY STATE v4.0 ===')
    if s.get('book_title'): print(f'  Book   : {s["book_title"]}')
    if s.get('genre'):      print(f'  Genre  : {s["genre"]}')
    print(f'  Chunks : {s["chunk_count"]} / {s["total_chunks"]}')
    print(f'  Setting: {s.get("current_setting","—")}')
    print(f'  Chars  : {len(s["characters"])} | Speech styles: {len(s["character_speech_styles"])}')
    for n, info in list(s['characters'].items())[:5]:
        style = s['character_speech_styles'].get(n, '—')
        print(f'    • {n} ({info.get("role","character")}) | speech: {style}')
    print(f'  Vocab  : {len(s["established_vocab"])} entries')
    if s['story_so_far']:
        last = s['story_so_far'][-1]
        txt  = last['summary'] if isinstance(last, dict) else str(last)
        print(f'  Last   : {txt[:90]}...')
    print('========================')


print('[OK] Story State Engine v4.0 loaded')
print('  NEW: character_speech_styles | lean context | 1-entry summary')
print('  Genre vocab: mystery/detective | gothic/horror | romance/social')
print('               adventure | folk tale / fairy tale | literary fiction')
print()
print('  Public functions: reset_for_new_book · set_genre · add_vocab_entry')
print('                    update_character(speech_style=...) · load_state · print_state_summary')


## 🔩 Step 6 — Load Translation Engine v9.2 (2-Pass)
**Run once.** Loads all prompts, scene detector, post-processors, and 2-pass Ollama engine.

**2-Pass Architecture:**
1. **Step 1 — Semantic Base**: Boring, precise, faithful Hindi (no style, no fillers)
2. **Step 2 — Style Filter**: Apply Hinglish tone safely on top of clean base

> This separates conflicting objectives: accuracy + style + tone + continuity no longer compete in one inference.


In [ ]:
import os, sys, json, time, warnings, re
from pathlib import Path
from datetime import datetime
warnings.filterwarnings('ignore')
from IPython.display import display, HTML, clear_output

# ════════════════════════════════════════════════════════════════════
# JARVIS DISPLAY HELPERS
# ════════════════════════════════════════════════════════════════════
def _j_bar(pct, w=180, fg='#c0392b', bg='#1a0505'):
    filled = max(0, min(int(w*(pct or 0)/100), w))
    return ('<div style="background:'+bg+';border:1px solid #2a1010;border-radius:3px;'
            'width:'+str(w)+'px;height:12px;display:inline-block;vertical-align:middle;">'
            '<div style="background:linear-gradient(90deg,#7b0000,'+fg+');'
            'width:'+str(filled)+'px;height:100%;border-radius:3px;'
            'box-shadow:0 0 7px '+fg+'88;"></div></div>')

def _j_col(u): return '#555' if u is None else ('#4CAF50' if u<40 else ('#FFD700' if u<75 else '#c0392b'))
def _j_fmt(s):
    s=max(0,int(s)); h,rem=divmod(s,3600); m,sec=divmod(rem,60)
    return (str(h)+'h '+str(m).zfill(2)+'m '+str(sec).zfill(2)+'s') if h else (str(m).zfill(2)+'m '+str(sec).zfill(2)+'s')
def _j_spark(vals):
    if not vals: return ''
    blk=' '+''.join(chr(c) for c in [9601,9602,9603,9604,9605,9606,9607,9608])
    mn,mx=min(vals),max(vals); rng=mx-mn or 1
    return ''.join(blk[min(8,int((v-mn)/rng*8))] for v in vals[-60:])
def _j_gpu():
    import subprocess
    try:
        out=subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu','--format=csv,noheader,nounits'],stderr=subprocess.DEVNULL,timeout=3).decode().strip().split(',')
        return float(out[0]),float(out[1])/1024,float(out[2])/1024,float(out[3])
    except: return None,None,None,None

def _jarvis_header(title, subtitle=''):
    display(HTML('<div style="background:linear-gradient(135deg,#0a0a0f,#1a0505);border:2px solid #c0392b;border-radius:8px;padding:14px 20px;font-family:Courier New,monospace;margin:6px 0;">'
                '<div style="color:#c0392b;font-size:1.2em;font-weight:bold;letter-spacing:3px;text-shadow:0 0 10px #c0392b;">'+title+'</div>'
                '<div style="color:#FFD700;font-size:0.8em;margin-top:4px;">'+subtitle+'</div>'
                '</div>'))
def _jarvis_ok(msg):
    display(HTML('<div style="font-family:Courier New,monospace;background:#0a0f0a;border-left:4px solid #4CAF50;padding:5px 14px;margin:2px 0;color:#4CAF50;font-size:0.88em;">[OK] '+str(msg)+'</div>'))
def _jarvis_info(msg, color='#FFD700'):
    display(HTML('<div style="font-family:Courier New,monospace;background:#0a0a0f;border-left:4px solid '+color+';padding:5px 14px;margin:2px 0;color:'+color+';font-size:0.88em;">[·] '+str(msg)+'</div>'))
def _jarvis_warn(msg):
    display(HTML('<div style="font-family:Courier New,monospace;background:#1a0505;border-left:4px solid #e74c3c;padding:5px 14px;margin:2px 0;color:#e74c3c;font-size:0.88em;">[!] '+str(msg)+'</div>'))


def _jarvis_chunk_dashboard(
        i, total, scene, in_words, in_chars, out_chars,
        chunk_t, total_t, eta_s, ctimes, errs,
        gpu_u, vr_u, vr_t, gpu_t,
        model_name, target_lang, tier, qa_passed, qa_summary_text='',
        context_chars=0, pass_label='2-pass'):
    pct    = i/total*100 if total else 0
    sc     = '#4CAF50' if pct>=100 else '#FFD700'
    vr_pct = (vr_u/vr_t*100) if (vr_u and vr_t) else 0
    gpu_u_d= str(int(gpu_u))+'%' if gpu_u is not None else 'N/A'
    vr_d   = (str(round(vr_u,1))+'/'+str(round(vr_t,1))+' GB') if vr_u else 'N/A'
    spark  = _j_spark(ctimes)
    t_avg  = str(round(sum(ctimes)/len(ctimes),1))+'s' if ctimes else '--'
    exp_s  = str(round(out_chars/in_chars,2))+'x' if in_chars else 'N/A'
    qa_col = '#4CAF50' if qa_passed else '#FFD700'
    qa_icon= '[OK]' if qa_passed else '[QA]'
    qa_txt = qa_summary_text if qa_summary_text else 'All checks passed'
    err_col= '#e74c3c' if errs else '#4CAF50'
    ctx_kb = f'{context_chars//1024}KB' if context_chars else 'n/a'
    p=[]
    p.append('<div style="background:#070710;border:2px solid #c0392b;border-radius:10px;font-family:Courier New,monospace;overflow:hidden;max-width:940px;">')
    p.append('<div style="background:linear-gradient(90deg,#1a0505,#0a0a1a,#1a0505);border-bottom:2px solid #c0392b;padding:10px 18px;display:flex;justify-content:space-between;align-items:center;">')
    p.append('<span style="color:#c0392b;font-size:1.2em;font-weight:bold;letter-spacing:4px;text-shadow:0 0 12px #c0392b;">J.A.R.V.I.S</span>')
    p.append('<span style="color:#2a2a3a;font-size:0.7em;">'+model_name+' | '+tier+' | '+pass_label+' | v9.2</span>')
    p.append('<span style="color:'+sc+';font-weight:bold;font-size:0.85em;">[ TRANSLATING ]</span>')
    p.append('</div>')
    p.append('<div style="padding:12px 18px;border-bottom:1px solid #1a1a2e;">')
    p.append('<div style="display:flex;justify-content:space-between;margin-bottom:5px;">')
    p.append('<span style="color:#FFD700;font-size:0.88em;">CHUNK <span style="color:#e8e8e8;font-size:1.1em;">'+str(i)+'</span><span style="color:#555;">/'+str(total)+'</span>&nbsp;&nbsp;<span style="color:#c0392b;">'+str(round(pct,1))+'%</span></span>')
    p.append('<span style="color:#555;font-size:0.78em;">scene: <span style="color:#e8e8e8;">'+scene+'</span> | '+str(in_words)+'w | expansion: <span style="color:#FFD700;">'+exp_s+'</span> | ctx: <span style="color:#4CAF50;">'+ctx_kb+'</span></span>')
    p.append('</div>')
    p.append('<div style="background:#12010a;border:1px solid #2a0a0a;border-radius:4px;padding:2px;">')
    p.append('<div style="background:linear-gradient(90deg,#5a0000,#a00010,#c0392b);height:18px;border-radius:3px;min-width:3px;box-shadow:0 0 10px #c0392b55;width:'+str(round(pct,2))+'%;"></div>')
    p.append('</div></div>')
    p.append('<div style="display:grid;grid-template-columns:repeat(5,1fr);border-bottom:1px solid #1a1a2e;">')
    for label,val in [('ELAPSED',_j_fmt(total_t)),('ETA',_j_fmt(eta_s)),
                       ('LAST',str(round(chunk_t,1))+'s'),('AVG',t_avg),
                       ('ERRORS','<span style="color:'+err_col+';">'+str(errs)+'</span>')]:
        p.append('<div style="background:#0d0d1a;padding:10px 12px;border-right:1px solid #1a1a2e;"><div style="color:#2a2a4a;font-size:0.63em;letter-spacing:2px;">'+label+'</div><div style="color:#FFD700;font-size:1em;font-weight:bold;">'+val+'</div></div>')
    p.append('</div>')
    p.append('<div style="display:grid;grid-template-columns:1fr 1fr;border-bottom:1px solid #1a1a2e;">')
    p.append('<div style="background:#0a0a18;padding:10px 14px;border-right:1px solid #1a1a2e;">'
             '<div style="color:#FFD700;font-size:0.63em;letter-spacing:2px;margin-bottom:6px;">GPU — TESLA T4</div>'
             '<div style="display:flex;align-items:center;gap:6px;margin-bottom:4px;">'
             '<span style="color:#2a2a4a;font-size:0.7em;width:52px;">UTIL</span>'
             +_j_bar(gpu_u,140,'#c0392b')+'<span style="color:'+_j_col(gpu_u)+';font-size:0.78em;">'+gpu_u_d+'</span></div>'
             '<div style="display:flex;align-items:center;gap:6px;">'
             '<span style="color:#2a2a4a;font-size:0.7em;width:52px;">VRAM</span>'
             +_j_bar(vr_pct,140,'#8B0000')+'<span style="color:#e8e8e8;font-size:0.78em;">'+vr_d+'</span></div>'
             '</div>')
    p.append('<div style="background:#0a0a18;padding:10px 14px;">'
             '<div style="color:#FFD700;font-size:0.63em;letter-spacing:2px;margin-bottom:6px;">STORY STATE ENGINE v4.0 | 2-PASS</div>'
             '<div style="color:'+qa_col+';font-size:0.8em;">'+qa_icon+' '+qa_txt+'</div>'
             '<div style="color:#2a2a4a;font-size:0.63em;margin-top:6px;">TIMING SPARK</div>'
             '<div style="color:#8B0000;font-family:monospace;font-size:0.88em;">'+(spark or '--')+'</div>'
             '<div style="color:#2a2a4a;font-size:0.63em;">avg '+t_avg+'</div>'
             '</div>')
    p.append('</div>')
    p.append('<div style="background:#050508;padding:4px 18px;"><span style="color:#18080a;font-size:0.63em;">STARK INDUSTRIES v9.2 | gemma3:27b | Story State v4.0 | 2-Pass | +ToneGuard +DeadBoring</span></div></div>')
    display(HTML(''.join(p)))


def _jarvis_complete(model_name, total_time, n_chunks, orig_chars, trans_chars,
                     deva_count, qa_failed, output_file, state_file=''):
    ratio = round(trans_chars/orig_chars,2) if orig_chars else 0
    qa_col= '#FFD700' if qa_failed else '#4CAF50'
    qa_txt= ('WARN '+str(qa_failed)+' chunks') if qa_failed else 'PASS — All clear'
    dv_col= '#e74c3c' if deva_count else '#4CAF50'
    dv_txt= ('WARN '+str(deva_count)+' chunks') if deva_count else 'PASS — Clean'
    out_nm= str(output_file).split('/')[-1]
    st_nm = str(state_file).split('/')[-1] if state_file else ''
    p=[]
    p.append('<div style="background:#070710;border:2px solid #4CAF50;border-radius:10px;font-family:Courier New,monospace;overflow:hidden;max-width:940px;margin-top:10px;">')
    p.append('<div style="background:linear-gradient(90deg,#0a1a0a,#0a0a1a,#0a1a0a);border-bottom:2px solid #4CAF50;padding:12px 20px;display:flex;justify-content:space-between;align-items:center;">')
    p.append('<span style="color:#4CAF50;font-size:1.25em;font-weight:bold;letter-spacing:3px;text-shadow:0 0 12px #4CAF50;">[ MISSION ACCOMPLISHED ]</span>')
    p.append('<span style="color:#2a4a2a;font-size:0.75em;">TRANSLATION COMPLETE — v9.2 gemma3:27b | 2-Pass</span></div>')
    p.append('<div style="display:grid;grid-template-columns:1fr 1fr;padding:16px 20px;gap:16px;">')
    p.append('<table style="color:#e8e8e8;font-size:0.88em;border-collapse:collapse;">')
    rows=[('Total Time',_j_fmt(total_time)),('Chunks',str(n_chunks)),
          ('Avg/Chunk',_j_fmt(total_time/n_chunks) if n_chunks else '--'),
          ('Input',format(orig_chars,',')+' chars'),('Output',format(trans_chars,',')+' chars'),
          ('Expansion',str(ratio)+'x')]
    for k,v in rows: p.append(f'<tr><td style="color:#FFD700;padding:3px 14px 3px 0;">{k}</td><td>{v}</td></tr>')
    p.append('</table><table style="color:#e8e8e8;font-size:0.88em;border-collapse:collapse;">')
    for k,c,v in [('Devanagari',dv_col,dv_txt),('QA Pipeline',qa_col,qa_txt)]:
        p.append(f'<tr><td style="color:#FFD700;padding:3px 14px 3px 0;">{k}</td><td style="color:{c};">{v}</td></tr>')
    p.append(f'<tr><td style="color:#FFD700;padding:3px 14px 3px 0;">Translation</td><td style="color:#888;font-size:0.85em;">{out_nm}</td></tr>')
    if st_nm: p.append(f'<tr><td style="color:#FFD700;padding:3px 14px 3px 0;">State JSON</td><td style="color:#888;font-size:0.85em;">{st_nm}</td></tr>')
    p.append('</table></div>')
    p.append('<div style="background:#050a05;border-top:1px solid #1a3a1a;padding:5px 20px;"><span style="color:#1a3a1a;font-size:0.65em;">STARK INDUSTRIES v9.0 | gemma3:27b | 2-Pass Architecture</span></div></div>')
    display(HTML(''.join(p)))


import torch
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
_gpu_nm  = torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'N/A'
_gpu_mem = f"{torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB" if DEVICE=='cuda' else 'N/A'
_jarvis_header('STARK INDUSTRIES — NEURAL TRANSLATION ARRAY v9.0',
               f'Device: {DEVICE.upper()} · {_gpu_nm} · {_gpu_mem} | Model: gemma3:27b | 2-Pass')


# ════════════════════════════════════════════════════════════════════
# SOURCE CLEANER
# ════════════════════════════════════════════════════════════════════
def _preprocess_english_source(text):
    """
    Clean the English source text BEFORE sending to translation model.
    Removes German (or other source-language) words that TranslateGemma
    failed to translate, and normalises pipeline header artifacts.
    """
    text = re.sub(
        r'^TRANSLATED TO ENGLISH\b.*?(?:={10,}|-{10,})\n+',
        '', text, flags=re.DOTALL
    )
    text = _strip_german_residuals(text)
    text = re.sub(r'[\u0900-\u097F]+', '', text)
    return text.strip()


def clean_source_text(text):
    text = re.sub(r'(?m)^[A-Z][A-Za-z \'\,\-]{3,}\s+by\s+[A-Z][A-Za-z .]{2,}\s+\d{1,4}\s*$', '', text)
    text = re.sub(r'(?m)^\s*Page\s+\d{1,4}\s*$', '', text)
    text = re.sub(r'(?m)^\s*\d{1,4}\s*$', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


# ════════════════════════════════════════════════════════════════════
# SCENE DETECTOR
# ════════════════════════════════════════════════════════════════════
_SCENE_KW = {
    'DEDUCTION':  ['deduce','observe','perceive','infer','conclude','obvious','evident','clue','evidence','reasoning','logically','therefore','analyse','examine','inspect','solve','mystery','suspect','motive'],
    'REVELATION': ['suddenly realised','suddenly realized','all became clear','it dawned','the truth','so it was','secret','revealed','discovered','uncovered','confession','admitted','at last','finally knew'],
    'TENSION':    ['crept','tiptoed','lurked','peered','silence fell','a shadow','locked','dare not','warning','danger','trap','ambush','suspicion','uneasy','dread','something wrong','followed'],
    'ACTION':     ['charged','attacked','fired','shot','chased','fled','fight','struck','wounded','escaped','panic','crashed','rushed','dashed','sprang','leaped','flung','grabbed','seized'],
    'EMOTIONAL':  ['wept','cried','sobbed','tears','grief','sorrow','mourning','heartbroken','tragedy','died','death','loss','loved','happiness','joy','reunion','farewell','embrace','forgive','regret'],
    'BANTER':     ['laughed','chuckled','smiled','grinned','teased','merry','amusing','absurd','funny','joke','wit','clever','nonsense','irony','sarcasm','playful','quipped','retorted'],
    'CONFRONTATION': ['how dare','you lied','you deceived','confess','deny','accused','confronted','guilty','crime','cornered','betrayed','traitor','fury','rage','demanded','shouted','threatened'],
    'HORROR':     ['monster','creature','beast','demon','ghost','horror','terror','nightmare','blood','corpse','grave','tomb','darkness','supernatural','curse','haunted'],
    'ROMANCE':    ['love','heart','desire','passion','longing','beautiful','blush','admire','enchanted','proposal','beloved','darling','dearest','kiss','embrace'],
    'PHILOSOPHICAL': ['meaning','purpose','existence','fate','destiny','justice','morality','conscience','soul','freedom','truth','wisdom','contemplate','ponder','reflect'],
    'DESCRIPTION': ['the room','the house','the garden','the street','morning','evening','sunset','dawn','rain','storm','wind','snow','fog','mountain','river','sea','forest'],
    'SPEECH':     ['my friends','let me explain','i shall','we must','i assure you','address','declare','proclaim','announce'],
}
def detect_scene_type(text):
    tl = text.lower()
    scores = {sc: sum(2 if ' ' in kw else 1 for kw in kws if kw in tl) for sc,kws in _SCENE_KW.items()}
    best = max(scores, key=scores.get)
    return 'DAILY_LIFE' if scores[best]==0 else best

SCENE_CTX = {
    'DEDUCTION':     'DEDUCTION — Precise, confident. Clue by clue. Holmes ki tarah.',
    'REVELATION':    'REVELATION — Short lines. Dramatic pause. Reader shocked.',
    'TENSION':       'TENSION — Slow. Cold. Short sentences. Dread build karo.',
    'ACTION':        'ACTION — Fast. Punchy. 4-5 words max per sentence.',
    'EMOTIONAL':     'EMOTIONAL — Slow, heavy. Use ... for pauses. Let it breathe.',
    'BANTER':        'BANTER — Light, witty, fun. Mazaa aaye.',
    'CONFRONTATION': 'CONFRONTATION — Direct, sharp. High stakes.',
    'SPEECH':        'SPEECH — Authority. Weight. Build-up.',
    'HORROR':        'HORROR — Dark, visceral, creepy. Reader uncomfortable.',
    'ROMANCE':       'ROMANCE — Warm, intimate. Feelings slowly unfold.',
    'PHILOSOPHICAL': 'PHILOSOPHICAL — Deep but accessible. Simple words, heavy meaning.',
    'DESCRIPTION':   'DESCRIPTION — Vivid, sensory. Take reader there.',
    'DAILY_LIFE':    'DAILY LIFE — Conversational, chill. Easy flow.',
}


# ════════════════════════════════════════════════════════════════════
# 2-PASS PROMPTS — v9.0
#
# STEP 1 — "BORING TRANSLATOR"
#   Goal: Get a stable, faithful Hindi base. No style. No personality.
#   Why: If Step1 is accurate, Step2 can't hallucinate — it has nowhere to drift.
#
# STEP 2 — "STYLE FILTER"
#   Goal: Apply Hinglish voice SAFELY on top of the clean base.
#   Key addition: STRICT FIDELITY RULE — tone cannot change.
#   Fix: Removed 'best friend' bias → 'naturally samjha rahe ho'
# ════════════════════════════════════════════════════════════════════

STEP1_SYSTEM = """You are a mechanical English-to-Hindi translator. Your only job is exact meaning transfer.

OUTPUT: ONLY the translated text. No labels, no preamble, no commentary.

STRICT RULES:
✓ Translate EVERY sentence completely — not one line missed
✓ Roman script ONLY — absolutely zero Devanagari characters
✓ Preserve the EXACT meaning of every sentence
✓ Preserve the EXACT emotional register — formal stays formal, serious stays serious, cold stays cold
✓ Preserve dialogue structure exactly — who says what to whom
✓ Keep sentence structure close to the original — do not restructure or reorder
✓ Paragraph breaks: follow the source exactly
✓ Gender agreement: male aaya/tha, female aayi/thi

HARD DO-NOT LIST (every line is a hard rule — no exceptions):
✗ Do NOT add tone, style, or personality of any kind
✗ Do NOT make it conversational — this is a translation, not a retelling
✗ Do NOT simplify or soften emotions — if source is cold, output must be cold
✗ Do NOT use GenZ words, fillers, or dramatic embellishment
✗ Do NOT add sarcasm, humor, or attitude unless the source text itself contains it
✗ Do NOT add opinions, commentary, or anything not explicitly in the source

If this output sounds boring and mechanical → that is CORRECT.
Boring = faithful. Faithful = the only goal of this stage.
First word of output = first translated word. Nothing else."""

STEP1_USER = """Translate this English text into simple, neutral Hindi (Roman script).
Your ONLY job is exact meaning transfer — nothing else.

Rules:
- Preserve meaning EXACTLY — not one word of intent lost
- Do NOT add tone, style, or personality
- Do NOT simplify or soften emotions — cold stays cold, tense stays tense, formal stays formal
- Do NOT make it conversational or friendly
- Keep sentence structure close to the original
- Boring, mechanical output = correct output for this stage

---
{chunk}
---

Hindi translation (Roman script only, first word = first translated word):"""


STEP2_SYSTEM = """Tum ek Hinglish style editor ho. Tumhare paas ek clean Hindi base translation hai.
Tumhara kaam hai: isko natural Hinglish mein convert karna — BINA emotional register badle.

Jaise kisi ko naturally samjha rahe ho — lekin kahani ka tone respect karo.

=== STRICT FIDELITY RULE — SABSE ZAROORI ===

Tum kisi bhi sentence ka emotional tone CHANGE NAHI KAR SAKTE.
Formal    → formal hi rahe
Serious   → serious hi rahe
Calm      → calm hi rahe
Angry     → angry hi rahe
Sad       → sad hi rahe

Tum SIRF language ko natural aur readable bana rahe ho.
Character ka personality, tone, ya register KABHI mat badlo.

GALAT: 'Please leave the room' → 'Bhai please nikal jao' (tone change — FAIL)
SAHI:  'Please leave the room' → 'Aap please room se bahar chale jaaiye' ✓

{tone_rules_section}

=== CHARACTER CONSISTENCY — HARD RULE ===

Neeche (USER message mein) CHARACTER SPEECH STYLES diye gaye hain.
Woh styles STRICT ORDERS hain — suggestions nahi.

Enforcement rules:
✗ Calm character → suddenly casual, aggressive, ya sarcastic NAHI banega
✗ Formal character → suddenly friendly, GenZ, ya streetwise NAHI banega
✗ Serious character → suddenly funny ya ironic NAHI banega
✓ Agar style 'calm, precise, analytical' hai → cold, measured, logical sentences
✓ Agar style 'warm, gentle' hai → nurturing, soft, polite sentences
✓ Agar style 'reserved, formal' hai → stiff, careful, distant sentences
✓ Character ka BOLNE KA ANDAAZ wahi rahega jo uska speech style batata hai
✓ Ek hi character → poori chunk mein ek hi tone — shift mat karo

=== CONTEXT CONTINUITY — ACTIVE ENFORCEMENT ===

Previous chunk ka tone is chunk mein bhi CONTINUE hoga.
USER message mein 'PICHLI TRANSLATION KA END' diya gaya hai — woh tone reference hai.

Rules:
✗ Previous chunk serious tha → iss chunk casual ya funny NAHI hoga
✗ Previous chunk formal tha → iss chunk relaxed ya slangy NAHI hoga
✓ Tone shift SIRF TAB karo jab source text mein clearly indicate ho
✓ Flow seamlessly continue karo — restart ya mood change mat karo
✓ Sentence length aur pace bhi maintain karo (ACTION fast rahe, EMOTIONAL slow rahe)


=== TONE GUARD — PROHIBITED BEHAVIORS ===

Yeh kaam KABHI MAT KARO — chahe kitna bhi "natural" lage:
✗ Sarcasm add mat karo — unless source text itself is sarcastic
✗ Attitude ya aggression mat daalo — unless source explicitly shows anger
✗ Street slang ya GenZ filler mat daalo — unless original character speaks that way
✗ Irony ya humor add mat karo — unless source text is ironic or humorous
✗ Emotional commentary mat daalo — "yaar kitna bura hua" type lines = FAIL
✗ Casual shortcuts mat lo — "bas" / "toh fir" type padding where source has none
✓ Style filter ka kaam = readability improvement ONLY, NOT creativity
✓ Agar source mein woh element nahi hai → output mein bhi nahi hoga

=== SIRF EK ABSOLUTE OUTPUT RULE ===
OUTPUT MEIN SIRF TRANSLATED HINGLISH TEXT.
Pehla word = pehla translated word. Koi prefix nahi. Koi label nahi.

=== ZAROORI RULES ===

R1 — ROMAN SCRIPT ONLY. Ek bhi Devanagari = FAIL.

R2 — HAR LINE TRANSLATE KARO. Ek bhi line skip = FAIL.

R3 — PARAGRAPH BREAKS: Har 2-3 sentences ke baad ek blank line.

R4 — ENGLISH SIRF YAHAN:
 • Proper nouns: character names, place names, book titles
 • Naturalized words: police, train, office, hotel, phone, doctor, car
 • Technical terms jinka Hinglish equivalent awkward ho
Baaki sab Hindi/Urdu mein.

R5 — RESPECT FORM: 'tu/tujhe/teri/tera/tune' kabhi nahi. Sirf 'tum/tumhe/tumhari/tumhara/tumne'.

R6 — GENDER: Male: aaya/tha/gaya | Female: aayi/thi/gayi

R7 — PASSIVE VOICE BANNED: 'mujhe le jaya gaya' → 'woh mujhe le gaye'

=== NATURAL VOICE — MILLENNIAL HINGLISH ===

FILLERS — jahaan naturally fit ho (force mat karo):
matlab · basically · obviously · seedha · phir bhi · waise · yaar

PAUSES — sentences ke ANDAR:
SAHI: 'Woh ruka... phir dheere se bola — samajh gaya.'

KABHI MAT KARO:
Apni khud ki reaction, opinion, ya commentary story mein mat daalo.
Jo source mein nahi hai, woh output mein bhi nahi hona chahiye.

=== DIALOGUE ===

Dialogue variety: bola · boli · chillaya · poochha · jawab diya · hansate hue bola · dheere se kaha · gusse mein bola

=== EXAMPLES ===

ACTION:
BASE: 'Cab se kood gaya. Driver ko paisa diya. Seedha ghar ke andar chala gaya.'
RIGHT: 'Cab se kooda. Driver ko paisa pakdaya. Seedha ghar ke andar bhaaga.'

FORMAL (FIDELITY RULE):
BASE: 'Kripya kamra chhod dijiye.'
RIGHT: 'Aap please room se bahar chale jaaiye.' ✓
WRONG: 'Bhai please nikal jao' ✗ (too casual — fidelity broken)

=== FINAL CHECK ===
Devanagari? = FAIL | tu/tujhe? = FAIL | Tone register changed? = FAIL
Character speech style violated? = FAIL | Previous tone broken? = FAIL
Sarcasm/attitude/slang added (not in source)? = FAIL
Paragraph breaks har 2-3 sentences? Check | Gender sahi? Check
Translation poori aur complete? Check

Poori translation ke BILKUL AAKHIR mein sirf yeh ek line daalo:
##PLOT_NOTE: [1 English sentence: main event of this chunk]"""

STEP2_USER = """{context_block}

SCENE TONE (reference only — output mein mat daalo):
{scene_context}

=== CHARACTER SPEECH STYLES — STRICTLY ENFORCE ===
(Har character ka tone/style FIXED hai — in rules ko CHANGE MAT KARO):
{char_styles}
(Upar diye styles se bahar jaana = FIDELITY VIOLATION — FAIL)

=== PREVIOUS CHUNK END — TONE REFERENCE ===
(Iss tone aur andaaz ko is chunk mein bhi CONTINUE karo — restart mat karo):
{overlap_section}

NEECHE DIYA BASE TRANSLATION KO HINGLISH MEIN STYLE KAR — EK LINE BHI MAT CHHODNA:
---
{base_translation}
---

Hinglish styled output (pehla word = pehla word. Koi extra text nahi):"""


# Legacy BASIC/INTERMEDIATE prompts (single-pass fallback)
TRANSLATION_PROMPTS = {
    'BASIC': {
        'system': 'You are a professional English-to-Hindi translator.\nOUTPUT: Only the Hinglish translation — nothing else.\nRULES: Translate ALL text · Simple words · Short sentences · Roman script · No Devanagari',
        'user': 'Translate to simple Hinglish. Roman script only.\n\n{context_block}\n\nEnglish:\n---\n{chunk}\n---\n\nHinglish:'
    },
    'INTERMEDIATE': {
        'system': 'You are an expert English-to-Hinglish translator.\n\nOUTPUT: ONLY the translated Hinglish text — nothing else.\n\nRULES:\n✓ Translate every sentence completely\n✓ Simple, modern Hindi vocabulary\n✓ Short, clear sentences (8-15 words avg)\n✓ Natural conversational flow\n✓ Roman script only — absolutely no Devanagari\n✓ Paragraph breaks every 2-3 sentences\n✓ Gender agreement: male aaya/tha, female aayi/thi\n✓ Always tum/tumhe, never tu/tujhe',
        'user': 'Translate to simple modern Hinglish. Roman script only. No Devanagari.\n\n{context_block}\n\nSCENE: {scene_context}\nPREVIOUS END: {overlap_section}\n\nEnglish:\n---\n{chunk}\n---\n\nHinglish (ONLY translated text):'
    },
}

LANG_NAMES = {
    'hinglish':'Hinglish','hin_Deva':'Hindi','ben_Beng':'Bengali',
    'tam_Taml':'Tamil','tel_Telu':'Telugu','mar_Deva':'Marathi','guj_Gujr':'Gujarati',
}


# ════════════════════════════════════════════════════════════════════
# CHUNKING
# ════════════════════════════════════════════════════════════════════
def chunk_text(text, chunk_words=450):
    paragraphs = re.split(r'\n\s*\n|\r\n\s*\r\n', text)
    paragraphs = [p.strip() for p in paragraphs if p.strip()]
    chunks, current_chunk, current_count = [], [], 0
    for para in paragraphs:
        pw = para.split(); pc = len(pw)
        if pc > chunk_words:
            if current_chunk:
                chunks.append('\n\n'.join(current_chunk))
                current_chunk, current_count = [], 0
            for i in range(0, len(pw), chunk_words):
                chunks.append(' '.join(pw[i:i+chunk_words]))
        elif current_count + pc > chunk_words and current_chunk:
            chunks.append('\n\n'.join(current_chunk))
            current_chunk, current_count = [para], pc
        else:
            current_chunk.append(para); current_count += pc
    if current_chunk: chunks.append('\n\n'.join(current_chunk))
    return chunks


# ════════════════════════════════════════════════════════════════════
# OVERLAP
# ════════════════════════════════════════════════════════════════════
def get_overlap(prev_translation, overlap_words=80):
    if not prev_translation or overlap_words == 0: return ''
    words = prev_translation.split()
    return ' '.join(words[-overlap_words:]) if len(words) > overlap_words else prev_translation

def build_overlap_section(prev_tail):
    if not prev_tail: return '(Pehla chunk — koi pichla context nahi)'
    return '[PICHLI TRANSLATION KA AAKHRI HISSA — DOBARA TRANSLATE MAT KARO]:\n' + prev_tail + '\n[YAHAN SE NAYI TRANSLATION SHURU KARO]'


# ════════════════════════════════════════════════════════════════════
# POST-PROCESSING
# ════════════════════════════════════════════════════════════════════
_H_ALLOW = {
    'nahi','kahi','bhai','yaar','karo','raha','wala','haan','abhi','acha','accha',
    'theek','sahi','dekho','suno','chalo','jao','aao','bolo','batao','kuch','kisi',
    'koi','sab','sabse','bahut','zyada','woh','yeh','uska','uski','mera','meri',
    'tumhara','tumhari','toh','bhi','tha','thi','the','kar','kiya','gaya','gayi',
    'gaye','aaya','aayi','aaye','pehle','baad','saath','andar','bahar','bilkul',
    'ekdum','pakka','lekin','magar','kyunki','isliye','jab','tab','agar','phir',
    'kabhi','poora','poori','thoda','thodi','matlab','basically','obviously','waise',
    'police','train','office','cab','hotel','station','phone','doctor','class',
    'sir','sahab','madam','mr','mrs','miss','dr','lord','lady','aur','main','maine',
    'usne','unhone','tumne','seedha','phir','fir','abhi',
}

def has_devanagari(text): return bool(re.search(r'[\u0900-\u097F]', text))


def extract_plot_note(text):
    """Strip ##PLOT_NOTE: from output and return (clean_text, english_summary)."""
    marker = '##PLOT_NOTE:'
    if marker in text:
        parts     = text.split(marker, 1)
        note_line = parts[1].strip().split('\n')[0].strip()
        note_line = re.sub(r'^[\[\(\*]+|[\]\)\*]+$', '', note_line).strip()[:250]
        return parts[0].strip(), note_line
    return text, ''


_COMMENTARY_STARTS = (
    'Yaar, yeh sab', 'Yaar, woh', 'Yaar, usko', 'Yaar, woh toh',
    'Matlab, woh', 'Matlab, usko', 'Matlab, yeh',
    'Obviously, woh', 'Obviously, usko',
    'Basically, woh', 'Basically, usko',
    'Seedha bolun toh', 'Seedha kahun toh', 'Seedha bol', 'Seedha bolu toh',
    'Woh toh seedha', 'Woh toh bas yeh',
)
_COMMENTARY_BODY = (
    'yaar. Matlab', 'yaar. Obviously', 'yaar. Basically',
    'matlab. Obviously', 'matlab. Basically',
)

def _strip_commentary_paragraphs(text):
    """
    Remove paragraphs inserted by the model that are NOT translations of source.
    Detects: (a) paragraphs starting with meta-commentary markers,
             (b) paragraphs that are purely meta-commentary (no dialogue, short),
             (c) standalone '...' or '—' lines between paragraphs.
    """
    paras = text.split('\n\n')
    clean = []
    for p in paras:
        s = p.strip()
        if not s:
            continue
        if s in ('...', '—', '–', '…'):
            continue
        if s.startswith('...') and any(m in s for m in
                ('Yaar', 'Matlab', 'Obviously', 'Basically', 'Seedha')):
            continue
        if any(s.startswith(m) for m in _COMMENTARY_STARTS):
            continue
        if s.startswith('—') and len(s.split()) < 20:
            continue
        words = s.split()
        if len(words) < 50 and not ('"' in s or '\u201c' in s or '\u201d' in s):
            meta_count = sum(1 for m in ('Yaar', 'Matlab', 'Obviously',
                                          'Basically', 'Seedha', 'seedha')
                             if m in s)
            if meta_count >= 2:
                continue
        clean.append(p)
    return '\n\n'.join(clean)


def _break_long_sentences(text, max_words=45):
    """
    Post-process to break sentences that are > max_words long.
    Breaks at coordinating conjunctions.
    """
    break_words = {'aur', 'lekin', 'magar', 'kyunki', 'isliye', 'toh', 'phir',
                   'jabki', 'halanki', 'warna', 'tab', 'jab'}
    result = []
    for para in text.split('\n'):
        words = para.split()
        if len(words) <= max_words:
            result.append(para)
            continue
        new_para = []
        current = []
        for i, w in enumerate(words):
            current.append(w)
            w_clean = w.lower().strip('.,;:!?"\'')
            if (len(current) >= max_words // 2 and
                    w_clean in break_words and
                    i < len(words) - 3):
                new_para.append(' '.join(current))
                current = []
        if current:
            new_para.append(' '.join(current))
        result.append('\n'.join(new_para))
    return '\n'.join(result)


_GERMAN_RESIDUALS = {
    r'\bMutter\b':        'Maa',
    r'\bVater\b':         'Papa',
    r'\bHerr\b':          'Mr.',
    r'\bFrau\b':          'Mrs.',
    r'\bFräulein\b':      'Miss',
    r'\bnaare stuhl\b':   'narrow chair',
    r'\bStuhl\b':         'chair',
    r'\bzisch\b':         'hiss',
    r'\bzischen\b':       'hissing',
    r'\bzischlaate\b':    'phunkaarate',
    r'\bzischend\b':      'phunkaate',
    r'\bGott\b':          'Bhagwan',
    r'\bJa\b':            'Haan',
    r'\bNein\b':          'Nahi',
    r'\bDanke\b':         'Shukriya',
    r'\bBitte\b':         'Please',
    r'\bAch\b':           'Ugh/Oho',
    r'\bghiseeda\b':      'ghisa hua',
    r'\bProcurator\b':    'Chief Clerk',
    r'\bProkurist\b':     'Chief Clerk',
    r'Mr\. Procurator':   'Mr. Chief Clerk',
    r'Mr\. Prokurist':    'Mr. Chief Clerk',
}

def _strip_german_residuals(text):
    for pattern, replacement in _GERMAN_RESIDUALS.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text


def clean_translation(text):
    """FIXED v9.0: _strip_commentary and _break_long_sentences now execute (were unreachable in v8.x)."""
    # Strip think tokens
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    text = re.sub(r'</?think>', '', text)
    # Strip Devanagari
    text = re.sub(r'[\u0900-\u097F]+', '', text)
    text = re.sub(r'  +', ' ', text)
    # Strip German residuals
    text = _strip_german_residuals(text)
    # Fix dialect errors
    text = re.sub(r'\bwohne\b', 'usne', text)
    text = re.sub(r'\bletah\b', 'leta', text)
    text = re.sub(r'\bwone\b',  'usne', text)
    text = re.sub(r'\bwoh ne\b', 'usne', text)
    # Strip markdown fences
    text = re.sub(r'```\w*\n?', '', text)
    text = re.sub(r'```', '', text)
    # Strip prefix labels
    text = re.sub(r'^(Translation:|Hindi Translation:|Hinglish Translation:|Here.s the translation:|Hinglish:|Hindi:)\s*',
                  '', text, flags=re.IGNORECASE | re.MULTILINE)
    # Strip context block bleed
    text = re.sub(r'=== STORY CONTEXT.*?=== CONTEXT KHATAM.*?===', '', text, flags=re.DOTALL)
    text = re.sub(r'\[PICHLI TRANSLATION.*?\[YAHAN SE NAYI.*?\]', '', text, flags=re.DOTALL)
    text = re.sub(r'\[PICHLI TRANSLATION.*?\[END\]', '', text, flags=re.DOTALL)
    text = re.sub(r'PICHLI TRANSLATION KA.*?\n', '', text, flags=re.DOTALL)
    # Strip separator artifacts
    text = re.sub(r'[\u2500-\u257F]{3,}', '', text)
    text = re.sub(r'(?m)^[=\-\*]{5,}\s*$', '', text)
    text = re.sub(r'(?m)^---\s*$', '', text)
    # FIX: Strip fake model commentary (now BEFORE return — was unreachable in v8.x)
    text = _strip_commentary_paragraphs(text)
    # FIX: Break overly long sentences (now BEFORE return — was unreachable in v8.x)
    text = _break_long_sentences(text)
    # Normalize whitespace
    lines = text.split('\n')
    cleaned = []
    for line in lines:
        s = line.strip()
        if s: cleaned.append(s)
        elif cleaned and cleaned[-1] != '': cleaned.append('')
    text = '\n'.join(cleaned)
    return re.sub(r'\n{3,}', '\n\n', text).strip()


def validate_translation(translated, prev_tail='', chunk_index=0, orig_text=''):
    issues = {}
    deva = re.findall(r'[\u0900-\u097F]+', translated)
    issues['devanagari'] = (False, f'{len(deva)} segment(s)') if deva else (True, 'Clean')
    seps = re.findall(r'[\u2500-\u257F]{3,}', translated)
    seps += re.findall(r'(?m)^[=\-\*]{5,}\s*$', translated)
    issues['separators'] = (False, f'{len(seps)}') if seps else (True, 'Clean')
    eng_runs = re.findall(r'\b[A-Za-z]{3,}(?:\s+[A-Za-z]{3,}){4,}\b', translated)
    real = [r for r in eng_runs
            if sum(1 for w in r.split() if re.match(r'^[a-z]{4,}$',w) and w not in _H_ALLOW) >= 3]
    issues['untranslated'] = (False, f'{len(real)} run(s)') if real else (True, 'Clean')
    if prev_tail and len(prev_tail) > 20:
        pw = set(w.lower() for w in prev_tail.split()[-15:])
        cw = set(w.lower() for w in translated.split()[:15])
        sim = len(pw & cw) / max(len(pw), 1)
        issues['overlap_dup'] = (False, f'High sim {sim:.0%}') if sim > 0.6 else (True, f'OK {sim:.0%}')
    else:
        issues['overlap_dup'] = (True, 'n/a')
    return issues


def summarise_issues(issues_dict):
    lines, ok = [], True
    for check, (passed, detail) in issues_dict.items():
        if not passed: ok = False
        lines.append(f'  {"✅" if passed else "⚠️ "} {check}: {detail}')
    return '\n'.join(lines), ok


# ════════════════════════════════════════════════════════════════════
# OLLAMA ENGINE v9.0 — 2-Pass Architecture
# ════════════════════════════════════════════════════════════════════
class OllamaTranslationEngine:
    """
    Gemma3:27b translation engine via Ollama — 2-Pass Architecture.

    Pass 1 — Semantic Base Translator:
      Input: English chunk
      Output: Clean, faithful Hindi (Roman script) — no style, no personality
      Why: Stable base layer prevents Step2 from hallucinating.

    Pass 2 — Style Filter:
      Input: Clean base translation
      Output: Natural Hinglish with STRICT FIDELITY RULE enforced
      Why: Model has only one job (style), not 6 competing objectives.

    ADVANCED tier = 2-pass (default).
    BASIC / INTERMEDIATE = single-pass (legacy, faster).
    """

    _OLLAMA_OPTIONS = {
        'temperature':    0.70,
        'top_k':          40,
        'top_p':          0.90,
        'repeat_penalty': 1.10,
        'num_predict':    2048,
        'num_ctx':        10240,
    }

    # Step 1 can run cooler — accuracy over creativity
    _OLLAMA_OPTIONS_STEP1 = {
        'temperature':    0.20,   # lower temp for faithful base
        'top_k':          20,
        'top_p':          0.85,
        'repeat_penalty': 1.05,
        'num_predict':    2048,
        'num_ctx':        10240,
    }

    def __init__(self, model_name, target_lang, tier='ADVANCED', num_ctx=8192,
                 tone_rules=None):
        self.model_name  = model_name
        self.target_lang = target_lang
        self.tier        = tier
        self.lang_name   = LANG_NAMES.get(target_lang, target_lang)
        self.tone_rules  = tone_rules or {}  # NEW v9.0: from BOOK_PROFILE
        self._OLLAMA_OPTIONS       = dict(self._OLLAMA_OPTIONS)
        self._OLLAMA_OPTIONS_STEP1 = dict(self._OLLAMA_OPTIONS_STEP1)
        self._OLLAMA_OPTIONS['num_ctx']       = num_ctx
        self._OLLAMA_OPTIONS_STEP1['num_ctx'] = num_ctx
        arch = '2-Pass (S1→S2)' if tier == 'ADVANCED' else 'Single-Pass'
        print(f'📥 Initializing Ollama engine: {model_name}')
        print(f'   Target: {self.lang_name} | Tier: {tier} | Architecture: {arch} | num_ctx: {num_ctx}')
        print(f'   Step1: temp=0.20 top_k=20 (faithful base)')
        print(f'   Step2: temp=0.70 top_k=40 (style filter)')
        try:
            import ollama
            self.client = ollama
            models = ollama.list()
            available = [m.get('name','').split(':')[0] for m in models.get('models',[])]
            base = model_name.split(':')[0]
            if not any(base in m for m in available):
                print(f'⚠️ {model_name} not found — pulling...')
                ollama.pull(model_name)
            else:
                print(f'✅ {model_name} ready!')
        except Exception as e:
            print(f'❌ Init error: {e}'); raise

    def _build_tone_rules_section(self):
        """Build tone rules injection string from BOOK_PROFILE tone_rules."""
        tr = self.tone_rules
        if not tr:
            return ''
        lines = ['=== BOOK TONE RULES (follow exactly) ===']
        if tr.get('default'):
            lines.append(f'Default tone: {tr["default"]}')
        if tr.get('avoid'):
            avoid_str = ', '.join(tr['avoid']) if isinstance(tr['avoid'], list) else tr['avoid']
            lines.append(f'AVOID: {avoid_str}')
        if tr.get('extra'):
            lines.append(f'Note: {tr["extra"]}')
        lines.append('')
        return '\n'.join(lines)

    def _build_char_styles_section(self):
        """
        Build character speech styles block for Step2 USER message.
        Format: Name → style | address: form — DO NOT change this style
        Header + footer add hard enforcement signal around the data.
        """
        styles = STORY_STATE.get('character_speech_styles', {})
        chars  = STORY_STATE.get('characters', {})
        if not chars:
            return '(No characters defined — engine will use neutral default style)'
        lines = ['Character speaking styles (FIXED — do NOT modify under any circumstances):']
        for name, info in list(chars.items())[:10]:
            style = styles.get(name, 'natural — keep neutral')
            addr  = info.get('address', 'aap')
            lines.append(f'  {name} → {style} | address: {addr} — DO NOT change this style')
        lines.append('Violating any of the above character styles = FIDELITY FAIL')
        return '\n'.join(lines)

    def _call_model(self, system, user, options, label=''):
        """Single Ollama API call with retry on failure."""
        response = self.client.chat(
            model   = self.model_name,
            messages= [
                {'role': 'system', 'content': system},
                {'role': 'user',   'content': user},
            ],
            options = options,
        )
        return response['message']['content']

    def translate(self, text, tier=None, prev_translation='', overlap_words=80,
                  context_block=''):
        """2-Pass translation for ADVANCED tier; single-pass for BASIC/INTERMEDIATE."""
        t = tier or self.tier
        if t == 'ADVANCED':
            return self._translate_two_pass(text, prev_translation, overlap_words, context_block)
        else:
            return self._translate_single_pass(text, t, prev_translation, overlap_words, context_block)

    def _translate_single_pass(self, text, tier, prev_translation, overlap_words, context_block):
        """Legacy single-pass for BASIC/INTERMEDIATE tiers."""
        prompts    = TRANSLATION_PROMPTS[tier]
        scene_type = detect_scene_type(text)
        scene_ctx  = SCENE_CTX.get(scene_type, SCENE_CTX['DAILY_LIFE'])
        overlap    = get_overlap(prev_translation, overlap_words)
        ovlp_sec   = build_overlap_section(overlap)
        system     = prompts['system']
        user       = prompts['user'].format(
            context_block   = context_block,
            scene_context   = scene_ctx,
            overlap_section = ovlp_sec,
            chunk           = text,
        )
        max_retries, last, plot_note_en = 3, None, ''
        for attempt in range(max_retries):
            try:
                raw = self._call_model(system, user, self._OLLAMA_OPTIONS)
                raw, _pn = extract_plot_note(raw)
                if _pn: plot_note_en = _pn
                translation = clean_translation(raw)
                last = translation
                if has_devanagari(translation) and attempt < max_retries - 1:
                    print(f'   ⚠️ Devanagari (attempt {attempt+1}), retrying...')
                    continue
                return translation, scene_type, plot_note_en
            except Exception as e:
                if attempt == max_retries - 1: raise
                print(f'   ⚠️ Attempt {attempt+1} failed: {e}')
        return last or '', scene_type, plot_note_en

    def _translate_two_pass(self, text, prev_translation, overlap_words, context_block):
        """
        2-Pass translation pipeline:
          Step 1: Boring, faithful base translation (low temp, no style)
          Step 2: Style filter — apply Hinglish voice with STRICT FIDELITY RULE
        """
        scene_type = detect_scene_type(text)
        scene_ctx  = SCENE_CTX.get(scene_type, SCENE_CTX['DAILY_LIFE'])
        overlap    = get_overlap(prev_translation, overlap_words)
        ovlp_sec   = build_overlap_section(overlap)
        plot_note_en = ''

        # ── STEP 1: Semantic Base Translation ────────────────────────
        step1_user = STEP1_USER.format(chunk=text)
        max_retries = 3
        base_translation = ''
        for attempt in range(max_retries):
            try:
                raw = self._call_model(STEP1_SYSTEM, step1_user, self._OLLAMA_OPTIONS_STEP1,
                                       label='Step1')
                raw = clean_translation(raw)
                if has_devanagari(raw) and attempt < max_retries - 1:
                    print(f'   ⚠️ Step1 Devanagari (attempt {attempt+1}), retrying...')
                    continue
                base_translation = raw
                break
            except Exception as e:
                if attempt == max_retries - 1:
                    print(f'   ⚠️ Step1 failed after {max_retries} attempts: {e}')
                    base_translation = text  # fallback: pass English through
                else:
                    print(f'   ⚠️ Step1 attempt {attempt+1} failed: {e}')

        # ── STEP 2: Style Filter ──────────────────────────────────────
        tone_rules_section = self._build_tone_rules_section()
        char_styles        = self._build_char_styles_section()
        step2_system = STEP2_SYSTEM.format(tone_rules_section=tone_rules_section)
        step2_user   = STEP2_USER.format(
            context_block    = context_block,
            scene_context    = scene_ctx,
            char_styles      = char_styles,
            overlap_section  = ovlp_sec,
            base_translation = base_translation,
        )
        final_translation = ''
        for attempt in range(max_retries):
            try:
                raw = self._call_model(step2_system, step2_user, self._OLLAMA_OPTIONS,
                                       label='Step2')
                raw, _pn = extract_plot_note(raw)
                if _pn: plot_note_en = _pn
                translation = clean_translation(raw)
                needs_retry = False
                if has_devanagari(translation):
                    print(f'   ⚠️ Step2 Devanagari (attempt {attempt+1}/{max_retries}), retrying...')
                    needs_retry = True
                if not needs_retry:
                    words = translation.split()
                    eng_c = sum(1 for w in words if re.match(r'^[a-z]{4,}$',w) and w not in _H_ALLOW)
                    if eng_c / max(len(words),1) > 0.55:
                        print(f'   ⚠️ Step2 high English ratio (attempt {attempt+1}), retrying...')
                        needs_retry = True
                if not needs_retry or attempt == max_retries - 1:
                    final_translation = translation
                    break
            except Exception as e:
                if attempt == max_retries - 1:
                    print(f'   ⚠️ Step2 failed: {e} — using Step1 base as output')
                    final_translation = base_translation
                else:
                    print(f'   ⚠️ Step2 attempt {attempt+1} failed: {e}')

        return final_translation, scene_type, plot_note_en


class OllamaTranslationGenerator:
    def __init__(self, model_name, target_lang, output_dir='.',
                 tier='ADVANCED', chunk_size=450, overlap_words=80, num_ctx=8192,
                 tone_rules=None):
        self.model_name    = model_name
        self.target_lang   = target_lang
        self.output_dir    = Path(output_dir)
        self.tier          = tier
        self.chunk_size    = chunk_size
        self.overlap_words = overlap_words
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.engine = OllamaTranslationEngine(
            model_name, target_lang, tier, num_ctx, tone_rules=tone_rules
        )

    def translate_file(self, input_file):
        arch = '2-Pass' if self.tier == 'ADVANCED' else 'Single-Pass'
        _jarvis_header(
            f'TRANSLATION GENERATOR v9.0 — Ollama / gemma3:27b | {arch}',
            f'Model: {self.model_name} | Tier: {self.tier} | Chunk: {self.chunk_size}w | Overlap: {self.overlap_words}w')
        with open(input_file, 'r', encoding='utf-8') as f:
            text = f.read()
        text = _preprocess_english_source(text)
        text = clean_source_text(text)
        orig_chars = len(text)
        _jarvis_info(f'📊 Input: {len(text.split()):,} words | {orig_chars:,} chars')
        chunks = chunk_text(text, self.chunk_size)
        STORY_STATE['total_chunks'] = len(chunks)
        est_lo = len(chunks) * 120 // 60  # 2-pass is ~2x slower
        est_hi = len(chunks) * 150 // 60
        _jarvis_ok(f'📦 {len(chunks)} chunks ({self.chunk_size}w target) | Est. time ({arch}): {est_lo}–{est_hi} min')

        translations, deva_flags = [], []
        start_time = time.time()
        _ctimes, _errs, _ewma = [], 0, None
        _ALPHA = 0.25
        qa_report = []
        timestamp  = datetime.now().strftime('%Y%m%d_%H%M%S')
        lang_code  = self.target_lang.split('_')[0]
        state_file = self.output_dir / f'state_{lang_code}_{timestamp}.json'

        for i, chunk in enumerate(chunks, 1):
            chunk_start = time.time()
            prev_trans  = translations[-1] if translations else ''
            prev_tail   = get_overlap(prev_trans, self.overlap_words) if prev_trans else ''

            # Build lean context BEFORE translating
            context_block = build_context_prompt()
            context_chars = len(context_block)

            try:
                translated, scene_type, plot_note_en = self.engine.translate(
                    chunk, self.tier, prev_trans, self.overlap_words, context_block)
                translated = clean_translation(translated)  # double-pass
                issues     = validate_translation(translated, prev_tail, i, chunk)
                qa_sum, all_ok = summarise_issues(issues)
                qa_report.append({'chunk': i, 'scene': scene_type, 'issues': issues})
                deva_flags.append(not issues['devanagari'][0])
                translations.append(translated)

                # Update Story State AFTER translating
                update_story_state(chunk, translated, i, scene_type, plot_note_en=plot_note_en)
                save_state(str(state_file))

                chunk_time = time.time() - chunk_start
                _ctimes.append(chunk_time)
                _ewma = chunk_time if _ewma is None else _ALPHA*chunk_time + (1-_ALPHA)*_ewma
                _eta  = _ewma * (len(chunks) - i)
                total_e = time.time() - start_time
                gpu_u, vr_u, vr_t, gpu_t = _j_gpu()
                qa_txt  = '' if all_ok else qa_sum.replace('\n',' | ')[:120]
                clear_output(wait=True)
                _jarvis_chunk_dashboard(
                    i, len(chunks), scene_type,
                    len(chunk.split()), len(chunk), len(translated),
                    chunk_time, total_e, _eta, _ctimes, _errs,
                    gpu_u, vr_u, vr_t, gpu_t,
                    self.model_name, self.target_lang, self.tier, all_ok, qa_txt, context_chars,
                    pass_label='2-Pass' if self.tier=='ADVANCED' else '1-Pass')
            except Exception as e:
                _errs += 1
                chunk_time = time.time() - chunk_start
                _ctimes.append(chunk_time)
                _ewma = chunk_time if _ewma is None else _ALPHA*chunk_time + (1-_ALPHA)*_ewma
                _eta  = _ewma * (len(chunks) - i)
                total_e = time.time() - start_time
                gpu_u, vr_u, vr_t, gpu_t = _j_gpu()
                translations.append(f'[ERROR chunk {i}: {e}]')
                deva_flags.append(False)
                qa_report.append({'chunk':i,'scene':'ERROR','issues':{'error':(False,str(e))}})
                clear_output(wait=True)
                _jarvis_chunk_dashboard(
                    i, len(chunks), 'ERROR',
                    len(chunk.split()), len(chunk), 0,
                    chunk_time, total_e, _eta, _ctimes, _errs,
                    gpu_u, vr_u, vr_t, gpu_t,
                    self.model_name, self.target_lang, self.tier, False, f'ERROR: {e}', 0)

        final_translation = '\n\n'.join(translations)
        output_file = self.output_dir / f'translation_{lang_code}_{timestamp}.txt'
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(final_translation)

        total_time  = time.time() - start_time
        trans_chars = len(final_translation)
        deva_count  = sum(deva_flags)
        qa_report_file = self.output_dir / f'qa_{lang_code}_{timestamp}.txt'
        qa_failed_chunks = []
        with open(qa_report_file, 'w', encoding='utf-8') as qf:
            qf.write(f'QA REPORT v9.0 — {timestamp}\nModel: {self.model_name}\nArchitecture: 2-Pass\n'+'='*60+'\n\n')
            for entry in qa_report:
                iss    = entry['issues']
                failed = [k for k,(p,_) in iss.items() if not p]
                status = 'PASS' if not failed else ('FAIL ['+ ', '.join(failed) +']')
                qf.write(f'Chunk {entry["chunk"]} ({entry["scene"]}): {status}\n')
                if failed:
                    qa_failed_chunks.append(entry['chunk'])
                    for k in failed: qf.write(f'  -> {k}: {iss[k][1]}\n')
            qf.write(f'\nSUMMARY: {len(qa_failed_chunks)}/{len(chunks)} chunks had issues\n')

        _jarvis_complete(self.model_name, total_time, len(chunks), orig_chars,
                         trans_chars, deva_count, len(qa_failed_chunks),
                         output_file, str(state_file))
        self.last_state_file = str(state_file)
        self.last_qa_file    = str(qa_report_file)
        return str(output_file), str(state_file)


_jarvis_ok('Translation Engine v9.2 loaded — 2-Pass Architecture')
_jarvis_ok('Step1: DEAD BORING base (temp=0.20, no style, no tone, no personality)')
_jarvis_ok('Step2: Style filter (temp=0.70, TONE GUARD active, char styles enforced)')
_jarvis_info('STRICT FIDELITY RULE: formal→formal, cold→cold, serious→serious')
_jarvis_info('TONE GUARD: sarcasm/attitude/slang blocked unless source contains it')
_jarvis_info('CHARACTER CONSISTENCY: Name → style | DO NOT change format enforced')
_jarvis_info('CONTEXT CONTINUITY: previous chunk tone maintained across boundary')
_jarvis_info('Story State v4.0: character_speech_styles | lean context | 1-entry summary')


## ⚡ Step 7 — Run Translation

**Only one thing to do:** fill in the `BOOK_PROFILE` dict at the top of this cell.

| Field | When to change it |
|-------|------------------|
| `title` + `genre` + `characters` + `extra_vocab` | Once per book |
| `tone_rules` | Once per book — controls STRICT FIDELITY RULE behavior |
| `resume_from` | Once per chapter (leave `''` for Ch1, paste state path for Ch2+, or `'auto'`) |

> **`tone_rules`** is the key new field. Set `default` to the book's overall register (e.g. `'formal, measured'`) and list what to `avoid` (e.g. `['sarcasm', 'street slang', 'attitude']`). The engine injects this into Step2.

> **`speech_style`** per character (4th element in the tuple) is optional but powerful — e.g. `'calm, precise, analytical'` for Holmes.

> **`resume_from = 'auto'`** automatically picks up the most recent downloaded state file.


In [ ]:
from IPython.display import display, HTML

# ════════════════════════════════════════════════════════════════════════
#   BOOK PROFILE  ←  THE ONLY SECTION YOU EDIT, ONCE PER BOOK
#
#   Fill in all five sections below before running.
#   For the next chapter of the same book: only change 'resume_from'.
# ════════════════════════════════════════════════════════════════════════

BOOK_PROFILE = {

    'title': 'The Metamorphosis',

'genre': 'literary fiction',

'characters': [
    ('Gregor', 'transformed protagonist', 'Thinks humanly but cannot speak, feels shame', 'woh'),
    ('Grete', 'younger sister', 'Initially caring, becomes resentful, plays violin', 'tum'),
    ('Father', 'head of household', 'Strict, aggressive, wears uniform, regained authority', 'aap'),
    ('Mother', 'frail matriarch', 'Asthmatic, protective but fearful, faints easily', 'aap'),
],

'extra_vocab': {
    'father': 'Papa',
    'mother': 'Maa',
    'sister': 'Didi',
    'living room': 'baithak',
    'front room': 'aagla kamra',
    'sweet milk': 'meetha doodh',
    'uniform': 'vardi',
    'asthma': 'dama',
    'conservatory': 'music conservatory',
    'guilders': 'paise',
    'safe': 'tijori',
    'nightgown': 'nightgown',
    'tonic': 'dawai',
    'maid': 'maid',
    'traveling salesman': 'traveling salesman',
    'business': 'business',
    'debt': 'karza',
    'apple': 'seb',
    'gas lamp': 'gas lamp',
    'newspaper': 'akhbaar',
},

'tone_rules': {
    'default': 'neutral, respectful, natural',
    'avoid':   ['sarcasm', 'attitude', 'street slang', 'GenZ filler words'],
    'extra':   '',   # leave '' if no extra instruction needed
},

'resume_from': '',

}


# ════════════════════════════════════════════════════════════════════════
#   NOTHING BELOW THIS LINE NEEDS TO CHANGE
# ════════════════════════════════════════════════════════════════════════

reset_for_new_book(title=BOOK_PROFILE['title'], genre=BOOK_PROFILE['genre'])

# NEW v9.0: 5-element character tuples (name, role, notes, addr, speech_style)
for char_entry in BOOK_PROFILE['characters']:
    if len(char_entry) == 5:
        name, role, notes, addr, speech_style = char_entry
    elif len(char_entry) == 4:
        name, role, notes, addr = char_entry
        speech_style = ''
    else:
        print(f'[WARN] Skipping malformed character entry: {char_entry}')
        continue
    update_character(name, role=role, notes=notes, address=addr,
                     speech_style=speech_style if speech_style else None)

for eng, hin in BOOK_PROFILE['extra_vocab'].items():
    add_vocab_entry(eng, hin)

_resume = BOOK_PROFILE['resume_from'].strip()
if _resume == 'auto':
    if not _load_latest_state(OUTPUT_DIR):
        print('[INFO] No previous state — starting fresh (Chapter 1 mode)')
elif _resume:
    load_state(_resume)

print()
print_state_summary()

display(HTML(f"""
<div style='background:linear-gradient(135deg,#0a0a0f,#1a0505);border:2px solid #c0392b;
            border-radius:8px;padding:16px 20px;font-family:Courier New,monospace;margin:12px 0;'>
  <div style='color:#c0392b;font-size:1.25em;font-weight:bold;letter-spacing:3px;'>JARVIS — PIPELINE v9.0 | 2-PASS</div>
  <table style='color:#e8e8e8;font-size:0.85em;margin-top:10px;border-collapse:collapse;'>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>MODEL</td><td>{MODEL}</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>TIER</td><td>{TRANSLATION_TIER} {'(2-pass)' if TRANSLATION_TIER=='ADVANCED' else '(single-pass)'}</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>BOOK</td>
        <td>{BOOK_PROFILE['title'] or 'not set'}</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>GENRE</td><td>{BOOK_PROFILE['genre']}</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>CHARACTERS</td>
        <td>{len(BOOK_PROFILE['characters'])}</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>TONE DEFAULT</td>
        <td style='color:#4CAF50;'>{BOOK_PROFILE['tone_rules'].get('default','—')}</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>AVOID</td>
        <td style='color:#e74c3c;font-size:0.82em;'>{', '.join(BOOK_PROFILE['tone_rules'].get('avoid',[]))}</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>VOCAB ENTRIES</td>
        <td>{len(BOOK_PROFILE['extra_vocab'])}</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>CHUNK / OVERLAP</td>
        <td>{CHUNK_SIZE}w / {OVERLAP_WORDS}w</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>RESUME</td>
        <td style='color:{"#4CAF50" if BOOK_PROFILE["resume_from"] else "#888"};'>
          {BOOK_PROFILE["resume_from"] or "fresh start"}</td></tr>
  </table>
</div>
"""))

print('Initializing v9.0 pipeline...')
generator = OllamaTranslationGenerator(
    model_name    = MODEL,
    target_lang   = TARGET_LANG,
    output_dir    = OUTPUT_DIR,
    tier          = TRANSLATION_TIER,
    chunk_size    = CHUNK_SIZE,
    overlap_words = OVERLAP_WORDS,
    num_ctx       = NUM_CTX,
    tone_rules    = BOOK_PROFILE['tone_rules'],  # NEW v9.0
)

print('\nStarting translation...')
OUTPUT_FILE, STATE_FILE = generator.translate_file(UPLOADED_FILE)

print(f'\nTranslation : {OUTPUT_FILE}')
print(f'State JSON  : {STATE_FILE}')
print()
print_state_summary()


## ⬇️ Step 8 — Download Outputs
Downloads Hinglish translation AND `state.json`.

In [ ]:
from google.colab import files
from IPython.display import display, HTML

display(HTML('<div style="background:#0a0f0a;border:2px solid #4CAF50;border-radius:8px;'
            'padding:12px 18px;font-family:Courier New,monospace;">'
            '<div style="color:#4CAF50;font-size:1.1em;font-weight:bold;">[>] OUTPUT EXTRACTION — v8.0 gemma3:27b</div>'
            '<div style="color:#888;font-size:0.82em;margin-top:4px;">Downloading translation + state.json</div></div>'))

print('📥 Downloading translation...')
files.download(OUTPUT_FILE)
print('📥 Downloading state.json (for multi-session resume)...')
files.download(STATE_FILE)

print('\n✅ Both files downloaded!')
print('\n💡 To resume Chapter 2 in a new session:')
print('   1. Run all setup cells (1–14)')
print('   2. In Cell 16, uncomment: load_state("path/to/state.json")')
print('   3. Upload Chapter 2 and run normally.')
print('   The engine will remember all characters, vocab, and story so far.')

## 💾 (Optional) Step 9 — Save to Google Drive

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/JARVIS_Translation'
os.makedirs(DRIVE_DIR, exist_ok=True)

shutil.copy(OUTPUT_FILE, os.path.join(DRIVE_DIR, os.path.basename(OUTPUT_FILE)))
shutil.copy(STATE_FILE,  os.path.join(DRIVE_DIR, os.path.basename(STATE_FILE)))

print(f'✅ Saved to: {DRIVE_DIR}')
print(f'   • {os.path.basename(OUTPUT_FILE)}')
print(f'   • {os.path.basename(STATE_FILE)}')